# Local-Rho Histograms and Free-Energy Estimates for the 20260408 Batch Comparison

This notebook mirrors the workflow of `python/analysis/analysis_NPH_20260324.ipynb`, but replaces the triangle-area analysis with a local-composition analysis based on

`rho = (N_A - N_B) / (N_A + N_B)`.

It compares the same nine simulation families:

- `results/20260309_NPH_batch_xyz_saving` -> `NPH`
- `results/20260313_NPH_batch_xyz_saving_piston` -> `NPH piston`
- `results/20260313_NVT_batch_xyz_saving` -> `NVT`
- `results/20260323_NPH_batch_xyz_saving_all_type0` -> `NPH unary`
- `results/20260323_NPH_batch_xyz_saving_piston_all_type0` -> `NPH piston unary`
- `results/20260323_NVT_batch_xyz_saving_all_type0` -> `NVT unary`
- `results/20260324_NPH_batch_xyz_saving` -> `NPH fast`
- `results/20260324_NPH_batch_xyz_saving_piston` -> `NPH piston fast`
- `results/20260324_NVT_batch_xyz_saving` -> `NVT fast`

For each simulation and temperature, it:

1. Streams the raw XYZ trajectory and keeps only the same second-phase frames used in the earlier notebook (`NPH`, `NPH_PISTON`, or `NVT_500`).
2. Reads the cutoff radius `rc` from the run config files under `configs/`.
3. For every particle in every second-phase frame, counts nearby `A` and `B` particles within `rc` using periodic boundary conditions.
4. Computes one per-frame histogram of `rho`, then averages those per-frame histograms across frames.
5. Writes one cached CSV per simulation and temperature.
6. Plots the `rho` histogram for each temperature in both a separated 3x3 view and an overlay view, using a logarithmic probability axis.
7. On the positive-rho side (`0 <= rho <= 1`), locates a lower-rho peak and a higher-rho peak, then estimates

   `delta E = -k_B T ln(P(rho_1) / P(rho_2))`.

Notes:

- By default this notebook uses reduced Lennard-Jones units with `k_B = 1`.
- The free-energy estimate uses the highest-probability bin in `[0, 0.5)` as `rho_1` and in `[0.5, 1]` as `rho_2`.
  Change `FREE_ENERGY_SPLIT` below if you want a different convention.


In [1]:
import json
import os
import re
from functools import lru_cache
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "python" / "particle_csv.py").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root containing python/particle_csv.py")


REPO_ROOT = find_repo_root()
ANALYSIS_ROOT = REPO_ROOT / "results" / "analysis_20260408_local_rho"
CSV_DIR = ANALYSIS_ROOT / "csv"
PLOTS_SEPARATE_DIR = ANALYSIS_ROOT / "plots_separate"
PLOTS_OVERLAY_DIR = ANALYSIS_ROOT / "plots_overlay"
PLOTS_SUMMARY_DIR = ANALYSIS_ROOT / "plots_summary"

for path in (ANALYSIS_ROOT, CSV_DIR, PLOTS_SEPARATE_DIR, PLOTS_OVERLAY_DIR, PLOTS_SUMMARY_DIR):
    path.mkdir(parents=True, exist_ok=True)

N_BINS = 50
RHO_BIN_EDGES = np.linspace(-1.0, 1.0, N_BINS + 1, dtype=np.float64)
FORCE_RECOMPUTE = False
FRAME_STRIDE = 1
MAX_SECOND_PHASE_FRAMES = None
BOLTZMANN_CONSTANT = 1.0
FREE_ENERGY_RHO_MIN = 0.0
FREE_ENERGY_SPLIT = 0.5
FREE_ENERGY_RHO_MAX = 1.0
FIG_DPI = 150

TEMP_PATTERN = re.compile(r"^T=(?P<value>[-+]?\d*\.?\d+)$")
XYZ_TAG_TO_TYPE = {"A": 0, "B": 1}

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = FIG_DPI
plt.rcParams["savefig.dpi"] = FIG_DPI
plt.rcParams["savefig.bbox"] = "tight"

RESULT_SPECS = [
    {
        "id": "nph",
        "label": "NPH",
        "cache_id": "nph_20260309",
        "root": REPO_ROOT / "results" / "20260309_NPH_batch_xyz_saving",
        "trajectory_name": "trajectory.xyz",
        "second_phase": "NPH",
        "color": "#1f77b4",
        "linestyle": "-",
    },
    {
        "id": "nph_piston",
        "label": "NPH piston",
        "cache_id": "nph_piston_20260313",
        "root": REPO_ROOT / "results" / "20260313_NPH_batch_xyz_saving_piston",
        "trajectory_name": "trajectory_piston.xyz",
        "second_phase": "NPH_PISTON",
        "color": "#d62728",
        "linestyle": "-",
    },
    {
        "id": "nvt",
        "label": "NVT",
        "cache_id": "nvt_20260313",
        "root": REPO_ROOT / "results" / "20260313_NVT_batch_xyz_saving",
        "trajectory_name": "trajectory_nvt.xyz",
        "second_phase": "NVT_500",
        "color": "#2ca02c",
        "linestyle": "-",
    },
    {
        "id": "nph_unary",
        "label": "NPH unary",
        "cache_id": "nph_all_type0_20260323",
        "root": REPO_ROOT / "results" / "20260323_NPH_batch_xyz_saving_all_type0",
        "trajectory_name": "trajectory.xyz",
        "second_phase": "NPH",
        "color": "#17becf",
        "linestyle": "--",
    },
    {
        "id": "nph_piston_unary",
        "label": "NPH piston unary",
        "cache_id": "nph_piston_all_type0_20260323",
        "root": REPO_ROOT / "results" / "20260323_NPH_batch_xyz_saving_piston_all_type0",
        "trajectory_name": "trajectory_piston.xyz",
        "second_phase": "NPH_PISTON",
        "color": "#ff7f0e",
        "linestyle": "--",
    },
    {
        "id": "nvt_unary",
        "label": "NVT unary",
        "cache_id": "nvt_all_type0_20260323",
        "root": REPO_ROOT / "results" / "20260323_NVT_batch_xyz_saving_all_type0",
        "trajectory_name": "trajectory_nvt.xyz",
        "second_phase": "NVT_500",
        "color": "#8c564b",
        "linestyle": "--",
    },
    {
        "id": "nph_fast",
        "label": "NPH fast",
        "cache_id": "nph_fast_20260324",
        "root": REPO_ROOT / "results" / "20260324_NPH_batch_xyz_saving",
        "trajectory_name": "trajectory.xyz",
        "second_phase": "NPH",
        "color": "#4e79a7",
        "linestyle": ":",
    },
    {
        "id": "nph_piston_fast",
        "label": "NPH piston fast",
        "cache_id": "nph_piston_fast_20260324",
        "root": REPO_ROOT / "results" / "20260324_NPH_batch_xyz_saving_piston",
        "trajectory_name": "trajectory_piston.xyz",
        "second_phase": "NPH_PISTON",
        "color": "#e15759",
        "linestyle": ":",
    },
    {
        "id": "nvt_fast",
        "label": "NVT fast",
        "cache_id": "nvt_fast_20260324",
        "root": REPO_ROOT / "results" / "20260324_NVT_batch_xyz_saving",
        "trajectory_name": "trajectory_nvt.xyz",
        "second_phase": "NVT_500",
        "color": "#59a14f",
        "linestyle": ":",
    },
]

SPEC_ORDER = {spec["id"]: idx for idx, spec in enumerate(RESULT_SPECS)}

print(f"Repo root: {REPO_ROOT}")
print(f"Histogram CSVs will be written under: {CSV_DIR}")
print(f"Separated plots will be written under: {PLOTS_SEPARATE_DIR}")
print(f"Overlay plots will be written under: {PLOTS_OVERLAY_DIR}")
print(f"Summary plots will be written under: {PLOTS_SUMMARY_DIR}")


def parse_temperature_dir_name(path: Path) -> float:
    match = TEMP_PATTERN.match(path.name)
    if not match:
        raise ValueError(f"Temperature directory must look like T=<value>, got {path.name!r}")
    return float(match.group("value"))


def safe_temp_label(temp_value: float) -> str:
    return f"{temp_value:.1f}".replace(".", "p")


def list_temperature_dirs(spec: dict) -> dict[float, Path]:
    mapping: dict[float, Path] = {}
    for temp_dir in sorted(spec["root"].glob("T=*")):
        if temp_dir.is_dir():
            mapping[parse_temperature_dir_name(temp_dir)] = temp_dir
    return mapping


def collect_temperature_values(specs: list[dict]) -> list[float]:
    values: set[float] = set()
    for spec in specs:
        values.update(list_temperature_dirs(spec).keys())
    return sorted(values)


def resolve_temperature_dir(spec: dict, temp_value: float) -> Path | None:
    temp_dir = spec["root"] / f"T={temp_value:.1f}"
    if temp_dir.is_dir():
        return temp_dir
    return None


def parse_metadata_line(line: str) -> dict[str, object]:
    parsed: dict[str, object] = {}
    for token in line.strip().split():
        key, value = token.split("=", 1)
        if key == "phase":
            parsed[key] = value
        elif key in {"global_step", "phase_step"}:
            parsed[key] = int(value)
        else:
            parsed[key] = float(value)
    return parsed


def parse_particle_type(tag: str) -> int:
    if tag not in XYZ_TAG_TO_TYPE:
        raise RuntimeError(f"Unexpected particle tag {tag!r} in XYZ data")
    return XYZ_TAG_TO_TYPE[tag]


def iter_xyz_frames(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        frame_index = 0
        while True:
            header = handle.readline()
            if not header:
                return

            header = header.strip()
            if not header:
                continue

            n_particles = int(header)
            metadata_line = handle.readline()
            if metadata_line == "":
                raise RuntimeError(f"Unexpected end of file after XYZ header in {path}")
            metadata = parse_metadata_line(metadata_line)

            positions = np.empty((n_particles, 2), dtype=np.float64)
            particle_types = np.empty(n_particles, dtype=np.int8)
            for idx in range(n_particles):
                particle_line = handle.readline()
                if particle_line == "":
                    raise RuntimeError(f"Unexpected end of file inside particle block for {path}")
                tag, x_str, y_str, _ = particle_line.split()
                positions[idx, 0] = float(x_str)
                positions[idx, 1] = float(y_str)
                particle_types[idx] = parse_particle_type(tag)

            yield {
                "frame_index": frame_index,
                "n_particles": n_particles,
                "phase": metadata["phase"],
                "global_step": metadata["global_step"],
                "phase_step": metadata["phase_step"],
                "global_time": metadata["global_time"],
                "phase_time": metadata["phase_time"],
                "Lx": metadata["Lx"],
                "Ly": metadata["Ly"],
                "positions": positions,
                "particle_types": particle_types,
            }
            frame_index += 1


def iter_second_phase_frames(
    spec: dict,
    temp_dir: Path,
    *,
    frame_stride: int = 1,
    max_frames: int | None = None,
):
    if frame_stride < 1:
        raise ValueError("frame_stride must be at least 1")

    second_phase_index = 0
    yielded = 0
    xyz_path = temp_dir / spec["trajectory_name"]

    for frame in iter_xyz_frames(xyz_path):
        if frame["phase"] != spec["second_phase"]:
            continue

        take_frame = (second_phase_index % frame_stride) == 0
        second_phase_index += 1
        if not take_frame:
            continue

        yield frame
        yielded += 1
        if max_frames is not None and yielded >= max_frames:
            return


@lru_cache(maxsize=None)
def load_cutoff_radius(temp_dir_str: str) -> tuple[float, tuple[str, ...]]:
    temp_dir = Path(temp_dir_str)
    config_dir = temp_dir / "configs"
    if not config_dir.is_dir():
        raise RuntimeError(f"Missing config directory: {config_dir}")

    cutoff_candidates: list[float] = []
    config_names: list[str] = []

    for config_path in sorted(config_dir.glob("*.json")):
        try:
            data = json.loads(config_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            continue

        if "cutoff" not in data:
            continue

        sigma_values = [
            float(data.get("SIGMA_AA", 1.0)),
            float(data.get("SIGMA_AB", 1.0)),
            float(data.get("SIGMA_BB", 1.0)),
        ]
        cutoff_candidates.append(float(data["cutoff"]) * max(sigma_values))
        config_names.append(config_path.name)

    if not cutoff_candidates:
        raise RuntimeError(f"Could not find a cutoff value in {config_dir}")

    return max(cutoff_candidates), tuple(config_names)


def csv_path_for_histogram(spec: dict, temp_value: float) -> Path:
    return CSV_DIR / f"local_rho_histogram_{spec['cache_id']}_T_{safe_temp_label(temp_value)}.csv"


def histogram_required_columns() -> list[str]:
    return [
        "temperature",
        "simulation_id",
        "simulation_label",
        "second_phase",
        "cutoff_radius",
        "frame_stride",
        "max_second_phase_frames",
        "n_particles",
        "n_frames",
        "first_phase_time",
        "last_phase_time",
        "mean_valid_particles_per_frame",
        "mean_nan_particles_per_frame",
        "mean_valid_fraction",
        "mean_neighbors_per_valid_particle",
        "config_files",
        "bin_index",
        "bin_left",
        "bin_right",
        "bin_center",
        "bin_width",
        "mean_count_per_frame",
        "mean_probability",
        "mean_density",
    ]


def csv_matches_settings(csv_path: Path, cutoff_radius: float) -> bool:
    if not csv_path.exists():
        return False

    df = pd.read_csv(csv_path)
    if df.empty:
        return False

    if not set(histogram_required_columns()).issubset(df.columns):
        return False

    csv_bin_edges = np.concatenate(
        [
            df["bin_left"].to_numpy(dtype=np.float64),
            df["bin_right"].to_numpy(dtype=np.float64)[-1:],
        ]
    )
    if len(csv_bin_edges) != len(RHO_BIN_EDGES) or not np.allclose(csv_bin_edges, RHO_BIN_EDGES):
        return False

    if not np.isclose(float(df["cutoff_radius"].iloc[0]), cutoff_radius):
        return False

    if int(df["frame_stride"].iloc[0]) != FRAME_STRIDE:
        return False

    stored_max_frames = df["max_second_phase_frames"].iloc[0]
    current_max_frames = np.nan if MAX_SECOND_PHASE_FRAMES is None else float(MAX_SECOND_PHASE_FRAMES)
    if pd.isna(stored_max_frames) != pd.isna(current_max_frames):
        return False
    if not pd.isna(stored_max_frames) and float(stored_max_frames) != float(current_max_frames):
        return False

    return True


def cell_grid_geometry(Lx: float, Ly: float, rc: float) -> tuple[int, int, float, float]:
    if not np.isfinite(Lx) or not np.isfinite(Ly) or Lx <= 0.0 or Ly <= 0.0:
        raise ValueError(f"Invalid box lengths Lx={Lx}, Ly={Ly}")
    if not np.isfinite(rc) or rc <= 0.0:
        raise ValueError(f"Invalid cutoff radius rc={rc}")

    n_cells_x = max(1, int(np.floor(Lx / rc)))
    n_cells_y = max(1, int(np.floor(Ly / rc)))
    cell_w = Lx / float(n_cells_x)
    cell_h = Ly / float(n_cells_y)
    return n_cells_x, n_cells_y, cell_w, cell_h


def compute_local_rho(
    positions: np.ndarray,
    particle_types: np.ndarray,
    Lx: float,
    Ly: float,
    rc: float,
) -> tuple[np.ndarray, np.ndarray]:
    positions = np.asarray(positions, dtype=np.float64)
    particle_types = np.asarray(particle_types, dtype=np.int8)

    n_particles = int(positions.shape[0])
    rho = np.full(n_particles, np.nan, dtype=np.float64)
    total_neighbors = np.zeros(n_particles, dtype=np.int32)
    if n_particles == 0:
        return rho, total_neighbors

    n_cells_x, n_cells_y, cell_w, cell_h = cell_grid_geometry(Lx, Ly, rc)
    wrapped_positions = positions.copy()
    wrapped_positions[:, 0] = np.mod(wrapped_positions[:, 0], Lx)
    wrapped_positions[:, 1] = np.mod(wrapped_positions[:, 1], Ly)

    ix = np.floor(wrapped_positions[:, 0] / cell_w).astype(np.int64)
    iy = np.floor(wrapped_positions[:, 1] / cell_h).astype(np.int64)
    ix = np.clip(ix, 0, n_cells_x - 1)
    iy = np.clip(iy, 0, n_cells_y - 1)

    flat_ids = ix + n_cells_x * iy
    order = np.argsort(flat_ids, kind="mergesort")
    sorted_flat_ids = flat_ids[order]
    unique_cells, start_idx, counts = np.unique(
        sorted_flat_ids,
        return_index=True,
        return_counts=True,
    )

    n_cells_total = n_cells_x * n_cells_y
    cell_start = np.full(n_cells_total, -1, dtype=np.int64)
    cell_count = np.zeros(n_cells_total, dtype=np.int64)
    cell_start[unique_cells] = start_idx
    cell_count[unique_cells] = counts

    rc2 = rc * rc

    for cell_id in unique_cells:
        start = int(cell_start[cell_id])
        count = int(cell_count[cell_id])
        center_idx = order[start : start + count]

        cx = int(cell_id % n_cells_x)
        cy = int(cell_id // n_cells_x)
        neighbor_cell_ids: list[int] = []
        seen: set[int] = set()

        for dy in (-1, 0, 1):
            ny = (cy + dy) % n_cells_y
            for dx in (-1, 0, 1):
                nx = (cx + dx) % n_cells_x
                neighbor_id = nx + n_cells_x * ny
                if neighbor_id in seen or cell_count[neighbor_id] == 0:
                    continue
                seen.add(neighbor_id)
                neighbor_cell_ids.append(neighbor_id)

        candidate_parts = [
            order[int(cell_start[neighbor_id]) : int(cell_start[neighbor_id]) + int(cell_count[neighbor_id])]
            for neighbor_id in neighbor_cell_ids
        ]
        candidate_idx = np.concatenate(candidate_parts)

        center_pos = wrapped_positions[center_idx]
        candidate_pos = wrapped_positions[candidate_idx]

        dx = candidate_pos[None, :, 0] - center_pos[:, None, 0]
        dx -= Lx * np.rint(dx / Lx)
        dy = candidate_pos[None, :, 1] - center_pos[:, None, 1]
        dy -= Ly * np.rint(dy / Ly)
        dist2 = dx * dx + dy * dy

        within_cutoff = dist2 <= rc2
        within_cutoff &= candidate_idx[None, :] != center_idx[:, None]

        candidate_is_a = particle_types[candidate_idx] == 0
        n_a = np.sum(within_cutoff & candidate_is_a[None, :], axis=1)
        n_b = np.sum(within_cutoff & (~candidate_is_a)[None, :], axis=1)
        neighbor_total = n_a + n_b

        total_neighbors[center_idx] = neighbor_total.astype(np.int32)
        valid_mask = neighbor_total > 0
        if np.any(valid_mask):
            rho[center_idx[valid_mask]] = (n_a[valid_mask] - n_b[valid_mask]) / neighbor_total[valid_mask]

    return rho, total_neighbors


def locate_probability_peak(
    bin_centers: np.ndarray,
    probability: np.ndarray,
    lower: float,
    upper: float,
    *,
    include_upper: bool = False,
) -> dict[str, float]:
    if include_upper:
        mask = (bin_centers >= lower) & (bin_centers <= upper) & (probability > 0.0)
    else:
        mask = (bin_centers >= lower) & (bin_centers < upper) & (probability > 0.0)

    if not np.any(mask):
        return {
            "rho_peak": np.nan,
            "probability_peak": np.nan,
        }

    mask_indices = np.flatnonzero(mask)
    local_index = int(np.argmax(probability[mask]))
    peak_index = int(mask_indices[local_index])
    return {
        "rho_peak": float(bin_centers[peak_index]),
        "probability_peak": float(probability[peak_index]),
    }


def estimate_free_energy_from_histogram(df: pd.DataFrame, temp_value: float) -> dict[str, float]:
    bin_centers = df["bin_center"].to_numpy(dtype=np.float64)
    probability = df["mean_probability"].to_numpy(dtype=np.float64)

    lower_peak = locate_probability_peak(
        bin_centers,
        probability,
        FREE_ENERGY_RHO_MIN,
        FREE_ENERGY_SPLIT,
        include_upper=False,
    )
    upper_peak = locate_probability_peak(
        bin_centers,
        probability,
        FREE_ENERGY_SPLIT,
        FREE_ENERGY_RHO_MAX,
        include_upper=True,
    )

    p1 = float(lower_peak["probability_peak"])
    p2 = float(upper_peak["probability_peak"])
    if not np.isfinite(p1) or not np.isfinite(p2) or p1 <= 0.0 or p2 <= 0.0:
        delta_free_energy = np.nan
    else:
        delta_free_energy = -BOLTZMANN_CONSTANT * float(temp_value) * np.log(p1 / p2)

    return {
        "rho_1_peak": float(lower_peak["rho_peak"]),
        "probability_rho_1_peak": p1,
        "rho_2_peak": float(upper_peak["rho_peak"]),
        "probability_rho_2_peak": p2,
        "delta_free_energy": float(delta_free_energy),
    }


def load_histogram_summary(
    spec: dict,
    temp_value: float,
    csv_path: Path,
    *,
    source: str,
) -> dict[str, object]:
    df = pd.read_csv(csv_path)
    free_energy_summary = estimate_free_energy_from_histogram(df, temp_value)

    return {
        "spec": spec,
        "temperature": temp_value,
        "csv_path": csv_path,
        "dataframe": df,
        "source": source,
        "n_frames": int(df["n_frames"].iloc[0]),
        "n_particles": int(df["n_particles"].iloc[0]),
        "cutoff_radius": float(df["cutoff_radius"].iloc[0]),
        "mean_valid_fraction": float(df["mean_valid_fraction"].iloc[0]),
        "mean_neighbors_per_valid_particle": float(df["mean_neighbors_per_valid_particle"].iloc[0]),
        "free_energy_summary": free_energy_summary,
    }


def compute_and_save_mean_histogram(spec: dict, temp_value: float, temp_dir: Path) -> dict[str, object]:
    cutoff_radius, config_files = load_cutoff_radius(str(temp_dir.resolve()))

    sum_counts = np.zeros(len(RHO_BIN_EDGES) - 1, dtype=np.float64)
    sum_probability = np.zeros(len(RHO_BIN_EDGES) - 1, dtype=np.float64)

    n_frames = 0
    n_particles: int | None = None
    first_phase_time: float | None = None
    last_phase_time: float | None = None

    valid_particles_total = 0
    nan_particles_total = 0
    valid_fraction_total = 0.0
    mean_neighbors_total = 0.0
    frames_with_valid_particles = 0

    for frame in iter_second_phase_frames(
        spec,
        temp_dir,
        frame_stride=FRAME_STRIDE,
        max_frames=MAX_SECOND_PHASE_FRAMES,
    ):
        if n_particles is None:
            n_particles = int(frame["n_particles"])
        elif n_particles != int(frame["n_particles"]):
            raise RuntimeError(f"Particle count changed within {temp_dir}")

        rho_values, neighbor_counts = compute_local_rho(
            frame["positions"],
            frame["particle_types"],
            float(frame["Lx"]),
            float(frame["Ly"]),
            cutoff_radius,
        )
        finite_mask = np.isfinite(rho_values)
        finite_rho = rho_values[finite_mask]

        counts, _ = np.histogram(finite_rho, bins=RHO_BIN_EDGES)
        sum_counts += counts
        if counts.sum() > 0:
            sum_probability += counts / counts.sum()

        valid_count = int(finite_mask.sum())
        nan_count = int((~finite_mask).sum())
        valid_particles_total += valid_count
        nan_particles_total += nan_count
        valid_fraction_total += valid_count / float(frame["n_particles"])

        if valid_count > 0:
            mean_neighbors_total += float(np.mean(neighbor_counts[finite_mask]))
            frames_with_valid_particles += 1

        if first_phase_time is None:
            first_phase_time = float(frame["phase_time"])
        last_phase_time = float(frame["phase_time"])
        n_frames += 1

    if n_frames == 0 or n_particles is None:
        raise RuntimeError(f"No second-phase frames were found for {spec['label']} at {temp_dir}")

    mean_count_per_frame = sum_counts / float(n_frames)
    mean_probability = sum_probability / float(n_frames)
    bin_width = np.diff(RHO_BIN_EDGES)
    mean_density = mean_probability / bin_width

    mean_valid_particles_per_frame = valid_particles_total / float(n_frames)
    mean_nan_particles_per_frame = nan_particles_total / float(n_frames)
    mean_valid_fraction = valid_fraction_total / float(n_frames)
    mean_neighbors_per_valid_particle = (
        np.nan
        if frames_with_valid_particles == 0
        else mean_neighbors_total / float(frames_with_valid_particles)
    )

    df = pd.DataFrame(
        {
            "temperature": temp_value,
            "simulation_id": spec["id"],
            "simulation_label": spec["label"],
            "second_phase": spec["second_phase"],
            "cutoff_radius": cutoff_radius,
            "frame_stride": FRAME_STRIDE,
            "max_second_phase_frames": np.nan
            if MAX_SECOND_PHASE_FRAMES is None
            else MAX_SECOND_PHASE_FRAMES,
            "n_particles": n_particles,
            "n_frames": n_frames,
            "first_phase_time": np.nan if first_phase_time is None else first_phase_time,
            "last_phase_time": np.nan if last_phase_time is None else last_phase_time,
            "mean_valid_particles_per_frame": mean_valid_particles_per_frame,
            "mean_nan_particles_per_frame": mean_nan_particles_per_frame,
            "mean_valid_fraction": mean_valid_fraction,
            "mean_neighbors_per_valid_particle": mean_neighbors_per_valid_particle,
            "config_files": ";".join(config_files),
            "bin_index": np.arange(len(RHO_BIN_EDGES) - 1, dtype=np.int32),
            "bin_left": RHO_BIN_EDGES[:-1],
            "bin_right": RHO_BIN_EDGES[1:],
            "bin_center": 0.5 * (RHO_BIN_EDGES[:-1] + RHO_BIN_EDGES[1:]),
            "bin_width": bin_width,
            "mean_count_per_frame": mean_count_per_frame,
            "mean_probability": mean_probability,
            "mean_density": mean_density,
        }
    )

    csv_path = csv_path_for_histogram(spec, temp_value)
    df.to_csv(csv_path, index=False)
    return load_histogram_summary(spec, temp_value, csv_path, source="computed")


def compute_or_load_mean_histogram(
    spec: dict,
    temp_value: float,
    temp_dir: Path,
    *,
    force_recompute: bool = False,
) -> dict[str, object]:
    cutoff_radius, _ = load_cutoff_radius(str(temp_dir.resolve()))
    csv_path = csv_path_for_histogram(spec, temp_value)
    if (not force_recompute) and csv_matches_settings(csv_path, cutoff_radius):
        return load_histogram_summary(spec, temp_value, csv_path, source="cached")
    return compute_and_save_mean_histogram(spec, temp_value, temp_dir)


def summarize_availability(specs: list[dict]) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for spec in specs:
        for temp_value, temp_dir in list_temperature_dirs(spec).items():
            xyz_path = temp_dir / spec["trajectory_name"]
            csv_path = csv_path_for_histogram(spec, temp_value)

            try:
                cutoff_radius, config_files = load_cutoff_radius(str(temp_dir.resolve()))
            except Exception:
                cutoff_radius, config_files = np.nan, tuple()

            rows.append(
                {
                    "temperature": temp_value,
                    "simulation_id": spec["id"],
                    "simulation_label": spec["label"],
                    "second_phase": spec["second_phase"],
                    "temp_dir": str(temp_dir.relative_to(REPO_ROOT)),
                    "xyz_exists": xyz_path.exists(),
                    "cached_csv_exists": csv_path.exists(),
                    "cached_csv_path": "" if not csv_path.exists() else str(csv_path.relative_to(REPO_ROOT)),
                    "cutoff_radius": cutoff_radius,
                    "config_files": ";".join(config_files),
                }
            )

    availability_df = pd.DataFrame(rows)
    if availability_df.empty:
        return availability_df

    availability_df["simulation_order"] = availability_df["simulation_id"].map(SPEC_ORDER)
    availability_df = availability_df.sort_values(["temperature", "simulation_order"]).reset_index(drop=True)
    return availability_df.drop(columns="simulation_order")


def analyze_temperature(
    temp_value: float,
    runs: list[dict[str, object]],
    *,
    force_recompute: bool = False,
) -> dict[str, object]:
    histograms: list[dict[str, object]] = []
    for run in runs:
        histograms.append(
            compute_or_load_mean_histogram(
                run["spec"],
                temp_value,
                run["temp_dir"],
                force_recompute=force_recompute,
            )
        )

    histograms.sort(key=lambda item: SPEC_ORDER[item["spec"]["id"]])
    return {
        "histograms": histograms,
    }


def histogram_stats_text(histogram: dict[str, object]) -> str:
    free_energy = float(histogram["free_energy_summary"]["delta_free_energy"])
    free_energy_text = "ΔE=nan" if not np.isfinite(free_energy) else f"ΔE={free_energy:.3f}"
    return "\n".join(
        [
            f"n_frames={histogram['n_frames']}",
            f"valid={histogram['mean_valid_fraction']:.3f}",
            f"rc={histogram['cutoff_radius']:.2f}",
            free_energy_text,
            histogram["source"],
        ]
    )


def separate_plot_output_path(temp_value: float) -> Path:
    return PLOTS_SEPARATE_DIR / f"local_rho_histograms_separate_T_{safe_temp_label(temp_value)}.svg"


def overlay_plot_output_path(temp_value: float) -> Path:
    return PLOTS_OVERLAY_DIR / f"local_rho_histograms_overlay_T_{safe_temp_label(temp_value)}.svg"


def free_energy_plot_output_path() -> Path:
    return PLOTS_SUMMARY_DIR / "local_rho_delta_free_energy_vs_temperature.svg"


def plot_rho_histograms_by_simulation(temp_value: float, result: dict[str, object]) -> Path:
    histogram_by_id = {histogram["spec"]["id"]: histogram for histogram in result["histograms"]}

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(15.0, 11.0),
        sharex=True,
        sharey=True,
        dpi=FIG_DPI,
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, spec in zip(axes, RESULT_SPECS):
        histogram = histogram_by_id.get(spec["id"])

        ax.set_title(spec["label"])
        ax.set_xlabel("rho")
        ax.set_ylabel("mean probability")
        ax.set_xlim(-1.0, 1.0)
        ax.set_yscale("log")
        ax.grid(alpha=0.3)
        ax.axvline(0.0, color="0.4", linestyle="--", linewidth=1.0, alpha=0.7)
        ax.axvline(FREE_ENERGY_SPLIT, color="0.5", linestyle=":", linewidth=1.0, alpha=0.7)

        if histogram is None:
            ax.text(
                0.5,
                0.5,
                "missing xyz",
                transform=ax.transAxes,
                ha="center",
                va="center",
                color="0.45",
                fontsize=11,
            )
            continue

        df = histogram["dataframe"]
        x = df["bin_center"].to_numpy(dtype=np.float64)
        y = df["mean_probability"].to_numpy(dtype=np.float64)
        positive_mask = y > 0.0

        if np.any(positive_mask):
            ax.plot(
                x[positive_mask],
                y[positive_mask],
                color=spec["color"],
                linestyle=spec["linestyle"],
                linewidth=2.0,
            )
        else:
            ax.text(
                0.5,
                0.5,
                "no plottable data",
                transform=ax.transAxes,
                ha="center",
                va="center",
                color="0.45",
                fontsize=11,
            )

        ax.text(
            0.98,
            0.95,
            histogram_stats_text(histogram),
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9,
        )

    fig.suptitle(
        f"Second-phase local-rho histogram comparison by simulation (T={temp_value:g})",
        y=1.02,
    )
    fig.tight_layout()

    output_path = separate_plot_output_path(temp_value)
    fig.savefig(output_path, dpi=FIG_DPI)
    display(fig)
    plt.close(fig)
    return output_path


def plot_rho_histograms_overlay(temp_value: float, result: dict[str, object]) -> Path | None:
    histograms = result["histograms"]
    if not histograms:
        return None

    histogram_by_id = {histogram["spec"]["id"]: histogram for histogram in histograms}
    fig, ax = plt.subplots(figsize=(10.5, 5.6), dpi=FIG_DPI)

    plotted_any = False
    for spec in RESULT_SPECS:
        histogram = histogram_by_id.get(spec["id"])
        if histogram is None:
            continue

        df = histogram["dataframe"]
        x = df["bin_center"].to_numpy(dtype=np.float64)
        y = df["mean_probability"].to_numpy(dtype=np.float64)
        positive_mask = y > 0.0
        if not np.any(positive_mask):
            continue

        ax.plot(
            x[positive_mask],
            y[positive_mask],
            label=spec["label"],
            color=spec["color"],
            linestyle=spec["linestyle"],
            linewidth=2.0,
        )
        plotted_any = True

    ax.set_title(f"Second-phase local-rho histogram overlay (T={temp_value:g})")
    ax.set_xlabel("rho")
    ax.set_ylabel("mean probability")
    ax.set_xlim(-1.0, 1.0)
    ax.set_yscale("log")
    ax.grid(alpha=0.3)
    ax.axvline(0.0, color="0.4", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.axvline(FREE_ENERGY_SPLIT, color="0.5", linestyle=":", linewidth=1.0, alpha=0.7)
    if plotted_any:
        ax.legend(frameon=False, ncol=3)
    else:
        ax.text(0.5, 0.5, "No plottable data", transform=ax.transAxes, ha="center", va="center")

    output_path = overlay_plot_output_path(temp_value)
    fig.savefig(output_path, dpi=FIG_DPI)
    display(fig)
    plt.close(fig)
    return output_path


def plot_delta_free_energy_vs_temperature(free_energy_df: pd.DataFrame) -> Path | None:
    if free_energy_df.empty:
        return None

    fig, ax = plt.subplots(figsize=(10.0, 5.4), dpi=FIG_DPI)
    plotted_any = False

    for spec in RESULT_SPECS:
        subset = free_energy_df[
            (free_energy_df["simulation_id"] == spec["id"])
            & np.isfinite(free_energy_df["delta_free_energy"])
        ].sort_values("temperature")
        if subset.empty:
            continue

        ax.plot(
            subset["temperature"].to_numpy(dtype=np.float64),
            subset["delta_free_energy"].to_numpy(dtype=np.float64),
            marker="o",
            markersize=5,
            color=spec["color"],
            linestyle=spec["linestyle"],
            linewidth=2.0,
            label=spec["label"],
        )
        plotted_any = True

    ax.axhline(0.0, color="0.35", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.set_title("Estimated local-rho free-energy difference vs temperature")
    ax.set_xlabel("temperature")
    ax.set_ylabel(r"$\Delta E = -k_B T \ln(P(\rho_1) / P(\rho_2))$")
    ax.grid(alpha=0.3)
    if plotted_any:
        ax.legend(frameon=False, ncol=3)
    else:
        ax.text(0.5, 0.5, "No valid free-energy estimates", transform=ax.transAxes, ha="center", va="center")

    output_path = free_energy_plot_output_path()
    fig.savefig(output_path, dpi=FIG_DPI)
    display(fig)
    plt.close(fig)
    return output_path


Repo root: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA
Histogram CSVs will be written under: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA/results/analysis_20260408_local_rho/csv
Separated plots will be written under: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA/results/analysis_20260408_local_rho/plots_separate
Overlay plots will be written under: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA/results/analysis_20260408_local_rho/plots_overlay
Summary plots will be written under: /nfs/roberts/scratch/pi_co54/bh692/MD_Simulation_CUDA/results/analysis_20260408_local_rho/plots_summary


In [2]:
availability_df = summarize_availability(RESULT_SPECS)
display(availability_df)

ordered_labels = [spec["label"] for spec in RESULT_SPECS]

if not availability_df.empty:
    availability_pivot = availability_df.pivot(
        index="temperature",
        columns="simulation_label",
        values="xyz_exists",
    ).reindex(columns=ordered_labels)
    display(availability_pivot)

    cache_pivot = availability_df.pivot(
        index="temperature",
        columns="simulation_label",
        values="cached_csv_exists",
    ).reindex(columns=ordered_labels)
    display(cache_pivot)

availability_csv_path = ANALYSIS_ROOT / "availability_summary.csv"
availability_df.to_csv(availability_csv_path, index=False)

all_temperature_values = collect_temperature_values(RESULT_SPECS)
available_runs_by_temp: dict[float, list[dict[str, object]]] = {}

for temp_value in all_temperature_values:
    runs: list[dict[str, object]] = []
    for spec in RESULT_SPECS:
        temp_dir = resolve_temperature_dir(spec, temp_value)
        if temp_dir is None:
            continue

        xyz_path = temp_dir / spec["trajectory_name"]
        if xyz_path.exists():
            runs.append({"spec": spec, "temp_dir": temp_dir})

    if runs:
        available_runs_by_temp[temp_value] = runs

cache_counts_by_temp = (
    availability_df.groupby("temperature")["cached_csv_exists"].sum().astype(int).to_dict()
    if not availability_df.empty
    else {}
)

availability_summary_rows = [
    {
        "temperature": temp_value,
        "n_available_simulations": len(runs),
        "n_cached_csvs": int(cache_counts_by_temp.get(temp_value, 0)),
        "simulation_labels": ", ".join(run["spec"]["label"] for run in runs),
    }
    for temp_value, runs in available_runs_by_temp.items()
]
availability_summary_df = pd.DataFrame(availability_summary_rows)
if not availability_summary_df.empty:
    availability_summary_df = availability_summary_df.sort_values("temperature").reset_index(drop=True)
display(availability_summary_df)

analysis_by_temp: dict[float, dict[str, object]] = {}
loaded_csv_paths: set[Path] = set()
written_csv_paths: set[Path] = set()

for temp_value, runs in available_runs_by_temp.items():
    print(
        f"Processing T={temp_value:g} with {len(runs)}/{len(RESULT_SPECS)} simulation(s) that have trajectory data..."
    )
    analysis_result = analyze_temperature(
        temp_value,
        runs,
        force_recompute=FORCE_RECOMPUTE,
    )
    analysis_by_temp[temp_value] = analysis_result

    for histogram in analysis_result["histograms"]:
        if histogram["source"] == "cached":
            loaded_csv_paths.add(histogram["csv_path"])
        else:
            written_csv_paths.add(histogram["csv_path"])

analysis_summary_rows: list[dict[str, object]] = []
free_energy_summary_rows: list[dict[str, object]] = []

for temp_value, result in analysis_by_temp.items():
    for histogram in result["histograms"]:
        free_energy = histogram["free_energy_summary"]

        summary_row = {
            "temperature": temp_value,
            "simulation_id": histogram["spec"]["id"],
            "simulation_label": histogram["spec"]["label"],
            "source": histogram["source"],
            "csv_path": str(histogram["csv_path"].relative_to(REPO_ROOT)),
            "n_frames": histogram["n_frames"],
            "n_particles": histogram["n_particles"],
            "cutoff_radius": histogram["cutoff_radius"],
            "mean_valid_fraction": histogram["mean_valid_fraction"],
            "mean_neighbors_per_valid_particle": histogram["mean_neighbors_per_valid_particle"],
            "rho_1_peak": free_energy["rho_1_peak"],
            "probability_rho_1_peak": free_energy["probability_rho_1_peak"],
            "rho_2_peak": free_energy["rho_2_peak"],
            "probability_rho_2_peak": free_energy["probability_rho_2_peak"],
            "delta_free_energy": free_energy["delta_free_energy"],
        }
        analysis_summary_rows.append(summary_row)
        free_energy_summary_rows.append(summary_row.copy())

analysis_summary_df = pd.DataFrame(analysis_summary_rows)
if not analysis_summary_df.empty:
    analysis_summary_df["simulation_order"] = analysis_summary_df["simulation_id"].map(SPEC_ORDER)
    analysis_summary_df = (
        analysis_summary_df.sort_values(["temperature", "simulation_order"])
        .drop(columns="simulation_order")
        .reset_index(drop=True)
    )
display(analysis_summary_df)

analysis_summary_csv_path = ANALYSIS_ROOT / "analysis_summary.csv"
analysis_summary_df.to_csv(analysis_summary_csv_path, index=False)

free_energy_summary_df = pd.DataFrame(free_energy_summary_rows)
if not free_energy_summary_df.empty:
    free_energy_summary_df["simulation_order"] = free_energy_summary_df["simulation_id"].map(SPEC_ORDER)
    free_energy_summary_df = (
        free_energy_summary_df.sort_values(["temperature", "simulation_order"])
        .drop(columns="simulation_order")
        .reset_index(drop=True)
    )
display(free_energy_summary_df)

free_energy_summary_csv_path = ANALYSIS_ROOT / "free_energy_summary.csv"
free_energy_summary_df.to_csv(free_energy_summary_csv_path, index=False)

if not free_energy_summary_df.empty:
    free_energy_pivot = free_energy_summary_df.pivot(
        index="temperature",
        columns="simulation_label",
        values="delta_free_energy",
    ).reindex(columns=ordered_labels)
    display(free_energy_pivot)

print("Loaded histogram CSVs from cache:")
if loaded_csv_paths:
    for path in sorted(loaded_csv_paths, key=lambda item: str(item)):
        print(" -", path.relative_to(REPO_ROOT))
else:
    print(" - none")

print("Newly written histogram CSVs:")
if written_csv_paths:
    for path in sorted(written_csv_paths, key=lambda item: str(item)):
        print(" -", path.relative_to(REPO_ROOT))
else:
    print(" - none")

saved_separate_paths: list[Path] = []
saved_overlay_paths: list[Path] = []

for temp_value, result in analysis_by_temp.items():
    saved_separate_paths.append(plot_rho_histograms_by_simulation(temp_value, result))
    overlay_path = plot_rho_histograms_overlay(temp_value, result)
    if overlay_path is not None:
        saved_overlay_paths.append(overlay_path)

print("Saved separated local-rho plots:")
for path in saved_separate_paths:
    print(" -", path.relative_to(REPO_ROOT))

print("Saved overlay local-rho plots:")
for path in saved_overlay_paths:
    print(" -", path.relative_to(REPO_ROOT))

free_energy_plot_path = plot_delta_free_energy_vs_temperature(free_energy_summary_df)
print("Free-energy summary CSV:", free_energy_summary_csv_path.relative_to(REPO_ROOT))
if free_energy_plot_path is not None:
    print("Free-energy summary plot:", free_energy_plot_path.relative_to(REPO_ROOT))


,temperature,simulation_id,simulation_label,second_phase,temp_dir,xyz_exists,cached_csv_exists,cached_csv_path,cutoff_radius,config_files
0,0.5,nph,NPH,NPH,results/20260309_NPH_batch_xyz_saving/T=0.5,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_large.json
1,0.5,nph_piston,NPH piston,NPH_PISTON,results/20260313_NPH_batch_xyz_saving_piston/T...,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_large_piston.input.json;config_large_pi...
2,0.5,nvt,NVT,NVT_500,results/20260313_NVT_batch_xyz_saving/T=0.5,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_large_piston.json
3,0.5,nph_unary,NPH unary,NPH,results/20260323_NPH_batch_xyz_saving_all_type...,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_nph.json;config_nvt.json;config_nvt_100...
4,0.5,nph_piston_unary,NPH piston unary,NPH_PISTON,results/20260323_NPH_batch_xyz_saving_piston_a...,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_nph_piston.json;config_nvt.json;config_...
5,0.5,nvt_unary,NVT unary,NVT_500,results/20260323_NVT_batch_xyz_saving_all_type...,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_nvt_100.input.json;config_nvt_100.json;...
6,0.5,nph_fast,NPH fast,NPH,results/20260324_NPH_batch_xyz_saving/T=0.5,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_nph.json;config_nvt.json
7,0.5,nph_piston_fast,NPH piston fast,NPH_PISTON,results/20260324_NPH_batch_xyz_saving_piston/T...,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_nph_piston.json;config_nvt.json
8,0.5,nvt_fast,NVT fast,NVT_500,results/20260324_NVT_batch_xyz_saving/T=0.5,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_nvt_100.json;config_nvt_500.json
9,0.6,nph,NPH,NPH,results/20260309_NPH_batch_xyz_saving/T=0.6,True,True,results/analysis_20260408_local_rho/csv/local_...,2.5,config_large.json


simulation_label,NPH,NPH piston,NVT,NPH unary,NPH piston unary,NVT unary,NPH fast,NPH piston fast,NVT fast
temperature,,,,,,,,,
0.5,True,True,True,True,True,True,True,True,True
0.6,True,True,True,True,True,True,True,True,True
0.7,True,True,True,True,True,True,True,True,True
0.8,True,True,True,True,True,True,True,True,True
0.9,True,True,True,True,True,True,True,True,True
1.0,True,True,True,True,True,True,True,True,True


simulation_label,NPH,NPH piston,NVT,NPH unary,NPH piston unary,NVT unary,NPH fast,NPH piston fast,NVT fast
temperature,,,,,,,,,
0.5,True,True,True,True,True,True,True,True,True
0.6,True,True,True,True,True,True,True,True,True
0.7,True,True,True,True,True,True,True,True,True
0.8,True,True,True,True,True,True,True,True,True
0.9,True,True,True,True,True,True,True,True,True
1.0,True,True,True,True,True,True,True,True,True


,temperature,n_available_simulations,n_cached_csvs,simulation_labels
0,0.5,9,9,"NPH, NPH piston, NVT, NPH unary, NPH piston un..."
1,0.6,9,9,"NPH, NPH piston, NVT, NPH unary, NPH piston un..."
2,0.7,9,9,"NPH, NPH piston, NVT, NPH unary, NPH piston un..."
3,0.8,9,9,"NPH, NPH piston, NVT, NPH unary, NPH piston un..."
4,0.9,9,9,"NPH, NPH piston, NVT, NPH unary, NPH piston un..."
5,1.0,9,9,"NPH, NPH piston, NVT, NPH unary, NPH piston un..."


Processing T=0.5 with 9/9 simulation(s) that have trajectory data...


KeyboardInterrupt: 